# Module 4 Worksheet — LangChain Internals: LCEL, Memory, Streaming
**Corrected in this version:** `InHouseLLM(...)` replaced with `get_chat_model(...)`, since the corrected wrapper uses LangChain's native `ChatOpenAI` directly instead of a custom LLM subclass — and it's correctly routed per model now.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

In [ ]:
# pip install langchain langchain-core --break-system-packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

## 1. Rebuilding RetrievalQA as LCEL (this module's teaser problem)
Compare to the `RetrievalQA.from_chain_type(...)` version from the earlier project.

In [ ]:
from rag_pure_python import SimpleVectorStore

store = SimpleVectorStore()
store.add(["MCP standardizes how LLMs call external tools through a client-server interface.",
           "RAG combines a retriever and a generator to ground LLM answers in context."])

def retrieve(question: str) -> str:
    chunks = [t for t, s in store.search(question, k=2)]
    return "\n".join(chunks)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context."),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])
llm = get_chat_model(model=MODEL_QWEN3_14B, max_tokens=200)

chain = (
    RunnableParallel(context=lambda x: retrieve(x["question"]), question=lambda x: x["question"])
    | prompt | llm | StrOutputParser()
)

print(chain.invoke({"question": "What is MCP?"}))

## 2. Streaming
Every Runnable supports `.stream()` — try it on the chain above. Since `get_chat_model()` returns a real `ChatOpenAI`, streaming support now depends on whether your serving stack (vLLM or similar) honors `stream=True` at the HTTP level, rather than on a custom `_call()` method that never streamed at all.

In [ ]:
print("Streaming output (tokens/chunks as they arrive):")
for chunk in chain.stream({"question": "What is MCP?"}):
    print(chunk, end="", flush=True)
print()

## 3. Conversation memory, three ways

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

history = [SystemMessage(content="You are a helpful assistant.")]

def chat_turn(user_text):
    history.append(HumanMessage(content=user_text))
    reply = llm.invoke(history).content
    history.append(AIMessage(content=reply))
    return reply

print(chat_turn("What's RAG?"))
print(chat_turn("Now explain it to a 10-year-old."))
print("\nFull buffer so far:", len(history), "messages")

## Teaser exercise
Implement a crude 'summary memory': after every 2 turns, ask the LLM to summarize the conversation so far into 1-2 sentences, and replace the raw history with that summary. Compare token cost vs the buffer approach above over a 10-turn conversation.